In [10]:
import os
import re
import json
import time
import hashlib
from dataclasses import dataclass, asdict
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

from playwright.sync_api import sync_playwright, TimeoutError as PWTimeoutError


In [11]:
BASE = "https://cbr.ru"
URL_MP_DEC = "https://cbr.ru/dkp/mp_dec/#t1"
URL_DECISION_KEY_RATE = "https://cbr.ru/dkp/mp_dec/decision_key_rate/#y2026"

OUT_DIR = "data_cbr_html_only"
RAW_DIR = os.path.join(OUT_DIR, "raw_html")
TEXT_PATH = os.path.join(OUT_DIR, "texts.jsonl")
META_PATH = os.path.join(OUT_DIR, "documents.jsonl")
ERRORS_PATH = os.path.join(OUT_DIR, "errors.jsonl")

UA = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
)


# -------------------------------------------------
# Data model
# -------------------------------------------------

@dataclass
class DocItem:
    doc_id: str
    section: str                # mp_decisions / decision_key_rate
    category: str               # press_release / summary / statement
    title: str
    published_at: str | None
    landing_url: str
    source_page: str

In [12]:
# -------------------------------------------------
# Utils
# -------------------------------------------------

def ensure_dirs():
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(RAW_DIR, exist_ok=True)


def norm_url(href: str) -> str:
    return urljoin(BASE, href)


def http_get(session: requests.Session, url: str, timeout=45) -> requests.Response:
    r = session.get(url, timeout=timeout, allow_redirects=True)
    r.raise_for_status()
    return r


def cleanup_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = "\n".join(line.strip() for line in text.splitlines())
    return text.strip()


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
        tag.decompose()

    main = soup.find("main")
    if not main:
        main = (
            soup.find("div", class_=re.compile(r"(content|page|article|main)", re.I))
            or soup
        )

    lines = []
    for el in main.find_all(["h1", "h2", "h3", "h4", "p", "li"]):
        txt = el.get_text(" ", strip=True)
        if not txt:
            continue
        if txt in {"Поделиться", "Наверх", "Скачать"}:
            continue
        lines.append(txt)

    return cleanup_text("\n".join(lines))


def save_raw_html(doc_id: str, html: str) -> str:
    path = os.path.join(RAW_DIR, f"{doc_id}.html")
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    return path


def write_jsonl(path: str, rows: list[dict]):
    if not rows:
        return
    with open(path, "a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def is_html_url(url: str) -> bool:
    path = urlparse(url).path.lower()
    return not re.search(r"\.(pdf|doc|docx|zip|xls|xlsx)$", path)


def parse_ru_date(text: str) -> str | None:
    """
    Возвращает дату как строку в исходном виде, если нашел что-то похожее:
    - 20 марта 2026 г.
    - 01.04.2026
    """
    if not text:
        return None

    text = " ".join(text.split())

    p1 = re.search(
        r"\b\d{1,2}\s+"
        r"(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)"
        r"\s+\d{4}\s*г?\.?\b",
        text,
        flags=re.I
    )
    if p1:
        return p1.group(0).strip()

    p2 = re.search(r"\b\d{2}\.\d{2}\.\d{4}\b", text)
    if p2:
        return p2.group(0).strip()

    return None


def classify_mp_dec_title(title: str) -> str | None:
    title = title.strip()
    if title.startswith("Банк России принял решение"):
        return "press_release"
    if title.startswith("Банк России сохранил"):
        return "press_release"
    if title.startswith("О ключевой ставке Банка России"):
        return "press_release"
    return None


def classify_decision_key_rate_title(title: str) -> str | None:
    title = title.strip()
    if title.startswith("Резюме обсуждения ключевой ставки"):
        return "summary"
    if title.startswith("Заявление Председателя Банка России"):
        return "statement"
    return None

In [13]:
# -------------------------------------------------
# Collectors
# -------------------------------------------------

def collect_mp_decisions_dynamic() -> list[DocItem]:
    """
    Страница:
      https://cbr.ru/dkp/mp_dec/#t1

    Нужно:
    - нажимать "Загрузить еще" пока кнопка есть;
    - брать только HTML-ссылки с заголовками:
        * Банк России принял решение ...
        * Банк России сохранил ...
        * О ключевой ставке Банка России ...
    - published_at брать из строки слева от заголовка.
    """
    items: list[DocItem] = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page(user_agent=UA)
        page.goto(URL_MP_DEC.split("#")[0], wait_until="networkidle", timeout=120_000)

        # Нажимаем "Загрузить еще" до конца
        max_clicks = 100
        for _ in range(max_clicks):
            clicked = False

            try:
                buttons = page.locator("text=Загрузить еще")
                count = buttons.count()

                for i in range(count):
                    btn = buttons.nth(i)
                    try:
                        if btn.is_visible():
                            btn.scroll_into_view_if_needed()
                            page.wait_for_timeout(500)
                            btn.click(timeout=10_000)
                            page.wait_for_timeout(1200)
                            clicked = True
                            break
                    except PWTimeoutError:
                        continue
                    except Exception:
                        continue

            except Exception:
                pass

            if not clicked:
                break

        html = page.content()
        browser.close()

    soup = BeautifulSoup(html, "lxml")
    seen = set()

    # Ищем все ссылки нужных типов, а дату пытаемся взять из ближайшего текстового контейнера
    for a in soup.select("a[href]"):
        title = a.get_text(" ", strip=True)
        if not title:
            continue

        category = classify_mp_dec_title(title)
        if category is None:
            continue

        href = norm_url(a.get("href", ""))
        if not href or not href.startswith(BASE) or not is_html_url(href):
            continue

        # На этой странице дата обычно лежит рядом в той же строке/карточке
        published_at = None

        # 1) пробуем взять текст ближайшего контейнера-родителя
        for parent in [a.parent, a.find_parent("div"), a.find_parent("li"), a.find_parent("article"), a.find_parent("tr")]:
            if parent:
                parent_text = parent.get_text(" ", strip=True)
                published_at = parse_ru_date(parent_text)
                if published_at:
                    break

        # 2) если не нашли, ищем в предыдущих соседях родительского контейнера
        if not published_at:
            row = a.find_parent(["div", "li", "article", "tr"])
            if row:
                prev_texts = []
                for prev in row.find_all_previous(limit=8):
                    txt = prev.get_text(" ", strip=True)
                    if txt:
                        prev_texts.append(txt)
                for txt in prev_texts:
                    published_at = parse_ru_date(txt)
                    if published_at:
                        break

        key = href
        if key in seen:
            continue
        seen.add(key)

        doc_id = hashlib.sha1(href.encode("utf-8")).hexdigest()
        items.append(
            DocItem(
                doc_id=doc_id,
                section="mp_decisions",
                category=category,
                title=title,
                published_at=published_at,
                landing_url=href,
                source_page=URL_MP_DEC.split("#")[0],
            )
        )

    return items


def _extract_items_from_decision_key_rate_html(html: str, source_page: str) -> list[DocItem]:
    """
    Парсим страницу Материалы по итогам заседаний... как обычный HTML.
    Нужны только:
    - Резюме обсуждения ключевой ставки
    - Заявление Председателя Банка России...
    Дату берем как ближайшую дату перед ссылкой.
    """
    soup = BeautifulSoup(html, "lxml")
    items: list[DocItem] = []
    seen = set()

    # Берем все текстовые и ссылочные узлы внутри main в порядке документа
    main = soup.find("main") or soup

    current_meeting = None
    last_date = None

    for node in main.descendants:
        # 1) Текстовые узлы: ловим даты и заголовок заседания
        if isinstance(node, str):
            txt = " ".join(node.split())
            if not txt:
                continue

            # Пример: "Заседание Совета директоров от 20.03.2026"
            m = re.search(r"Заседание Совета директоров от (\d{2}\.\d{2}\.\d{4})", txt)
            if m:
                current_meeting = m.group(1)
                last_date = None
                continue

            d = parse_ru_date(txt)
            if d:
                last_date = d
                continue

        # 2) Ссылки: берем только нужные типы
        if getattr(node, "name", None) == "a" and node.has_attr("href"):
            title = node.get_text(" ", strip=True)
            if not title:
                continue

            category = classify_decision_key_rate_title(title)
            if category is None:
                continue

            href = norm_url(node.get("href", ""))
            if not href or not href.startswith(BASE) or not is_html_url(href):
                continue

            # Логика дат:
            # summary -> дата публикации обычно стоит прямо перед ссылкой (например 01.04.2026)
            # statement -> тоже дата публикации прямо перед ссылкой (часто дата заседания)
            published_at = last_date

            # запасной вариант
            if not published_at:
                parent_text = node.parent.get_text(" ", strip=True) if node.parent else ""
                published_at = parse_ru_date(parent_text)

            # еще один запасной вариант: дата заседания блока
            if not published_at:
                published_at = current_meeting

            key = href
            if key in seen:
                continue
            seen.add(key)

            doc_id = hashlib.sha1(href.encode("utf-8")).hexdigest()
            items.append(
                DocItem(
                    doc_id=doc_id,
                    section="decision_key_rate",
                    category=category,
                    title=title,
                    published_at=published_at,
                    landing_url=href,
                    source_page=source_page,
                )
            )

    return items



def collect_decision_key_rate_dynamic() -> list[DocItem]:
    """
    На этой странице Playwright не нужен:
    все нужные ссылки и даты уже есть в HTML.
    """
    session = requests.Session()
    session.headers.update({
        "User-Agent": UA,
        "Accept-Language": "ru,en;q=0.8",
    })

    html = http_get(session, URL_DECISION_KEY_RATE.split("#")[0]).text
    return _extract_items_from_decision_key_rate_html(
        html=html,
        source_page=URL_DECISION_KEY_RATE.split("#")[0]
    )

In [14]:
# -------------------------------------------------
# Processing pages
# -------------------------------------------------

def process_item(session: requests.Session, item: DocItem) -> tuple[dict, dict]:
    html = http_get(session, item.landing_url).text
    raw_path = save_raw_html(item.doc_id, html)
    text_main = html_to_text(html)

    meta = asdict(item)
    meta["raw_path"] = raw_path
    meta["text_len"] = len(text_main)

    txt = {
        "doc_id": item.doc_id,
        "section": item.section,
        "category": item.category,
        "title": item.title,
        "published_at": item.published_at,
        "landing_url": item.landing_url,
        "source_page": item.source_page,
        "text_main": text_main,
    }
    return meta, txt

In [15]:
def main():
    ensure_dirs()

    # чтобы не дописывать в старые файлы при повторном запуске
    for path in [TEXT_PATH, META_PATH, ERRORS_PATH]:
        if os.path.exists(path):
            os.remove(path)

    session = requests.Session()
    session.headers.update({
        "User-Agent": UA,
        "Accept-Language": "ru,en;q=0.8",
    })

    print("Collect MP decisions (dynamic, click 'Загрузить еще') ...")
    items_mp = collect_mp_decisions_dynamic()

    print("Collect decision key rate materials (dynamic, tabs 2026/2025/2024) ...")
    items_dkr = collect_decision_key_rate_dynamic()

    all_items = items_mp + items_dkr

    # финальный dedup
    uniq = {}
    for item in all_items:
        uniq[item.landing_url] = item
    all_items = list(uniq.values())

    print(f"Collected HTML pages: {len(all_items)}")

    meta_rows = []
    text_rows = []

    for item in tqdm(all_items, desc="Download HTML + extract text"):
        try:
            meta, txt = process_item(session, item)
            meta_rows.append(meta)
            text_rows.append(txt)

            if len(meta_rows) >= 20:
                write_jsonl(META_PATH, meta_rows)
                write_jsonl(TEXT_PATH, text_rows)
                meta_rows.clear()
                text_rows.clear()

        except Exception as e:
            err = asdict(item)
            err["error"] = str(e)
            write_jsonl(ERRORS_PATH, [err])

    write_jsonl(META_PATH, meta_rows)
    write_jsonl(TEXT_PATH, text_rows)

    print("Done.")
    print(f"Metadata: {META_PATH}")
    print(f"Texts:    {TEXT_PATH}")
    print(f"Raw HTML: {RAW_DIR}")
    print(f"Errors:   {ERRORS_PATH}")

In [ ]:
if __name__ == "__main__":
    main()

In [17]:
import sys
print(sys.executable)

C:\Users\user\AppData\Local\Programs\Python\Python313\python.exe
